# 02 — Kaggle Balanced Preprocessing Dataset Builder

This standalone Kaggle notebook builds the processed dataset required by the RoBERTa/LoRA notebook. It does not depend on the local project `src/` package. It removes exact duplicates, samples an equal number of ratings `1..5`, applies light Transformer-safe preprocessing, merges product metadata, creates the selected `model_input`, performs a stratified train/validation split, validates every output, and creates a ZIP ready to download or publish as a Kaggle Dataset.

The official test dataset is not read or modified.

## Kaggle instructions

1. Add the raw dataset containing `train_data.csv` and `title_brand.csv` as Notebook input.
2. Set `KAGGLE_DATASET_SLUG` only if automatic discovery is ambiguous.
3. Change only `SAMPLES_PER_CLASS` to scale the experiment.
4. Run all cells.
5. Download the generated ZIP or save a Kaggle Notebook version and publish the output as a new Kaggle Dataset.

In [ ]:
from pathlib import Path
import json
import re
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

SAMPLES_PER_CLASS = 50_000  # Total selected rows per rating class.
VALIDATION_PER_CLASS = 2_000  # Keep validation fixed while scaling training data.
RANDOM_STATE = 42
TARGET_COLUMN = 'overall'
TEXT_COLUMN = 'reviewText'
SUMMARY_MAX_WORDS = 40  # Reserve the remaining token budget for reviewText.

# Example: 'amazon-review-raw-data'. Keep None for automatic discovery.
KAGGLE_DATASET_SLUG = None

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = (
    Path('/kaggle/working')
    / f'bert_balanced_{SAMPLES_PER_CLASS}_per_class'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## Locate the raw Kaggle files

In [ ]:
def find_unique_file(filename, required_columns):
    search_root = INPUT_ROOT / KAGGLE_DATASET_SLUG if KAGGLE_DATASET_SLUG else INPUT_ROOT
    candidates = sorted(search_root.rglob(filename))
    valid = []
    for path in candidates:
        try:
            columns = set(pd.read_csv(path, nrows=2).columns)
            if set(required_columns).issubset(columns):
                valid.append(path)
        except Exception:
            continue
    if len(valid) != 1:
        raise FileNotFoundError(
            f'Expected exactly one valid {filename}; found {valid}. '
            'Set KAGGLE_DATASET_SLUG explicitly.'
        )
    return valid[0]

TRAIN_PATH = find_unique_file(
    'train_data.csv', ['overall', 'reviewText', 'summary', 'asin']
)
TEST_PATH = find_unique_file(
    'test_data.csv', ['reviewText', 'summary', 'verified', 'vote', 'asin']
)
PRODUCT_PATH = find_unique_file(
    'title_brand.csv', ['asin', 'title', 'brand']
)
print('Training data:', TRAIN_PATH)
print('Test data:', TEST_PATH)
print('Product metadata:', PRODUCT_PATH)

## Transformer-safe text cleaning

HTML and URLs are removed and whitespace is normalized. Case, punctuation, stopwords, negation, numbers, emoji, and natural sentence structure are preserved.

In [ ]:
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r'\s+')

def preprocess_for_transformer(value):
    if not isinstance(value, str):
        return ''
    text = value
    if '<' in text and '>' in text:
        text = BeautifulSoup(text, 'html.parser').get_text(separator=' ')
    text = URL_PATTERN.sub('', text)
    return WHITESPACE_PATTERN.sub(' ', text).strip()

assert preprocess_for_transformer('<b>Not bad!</b> https://example.com') == 'Not bad!'

## Load, deduplicate, and create the balanced sample

In [ ]:
raw_df = pd.read_csv(TRAIN_PATH, low_memory=False)
required_columns = {TARGET_COLUMN, TEXT_COLUMN, 'summary', 'verified', 'vote', 'asin'}
missing_columns = required_columns.difference(raw_df.columns)
assert not missing_columns, f'Missing required columns: {sorted(missing_columns)}'

raw_rows = len(raw_df)
deduplicated_df = raw_df.drop_duplicates().reset_index(drop=True)
exact_duplicates_removed = raw_rows - len(deduplicated_df)
cleaned_review_candidate = (
    deduplicated_df[TEXT_COLUMN].fillna('').astype(str).map(preprocess_for_transformer)
)
eligible_review_mask = cleaned_review_candidate.str.strip().ne('')
empty_source_reviews_removed = int((~eligible_review_mask).sum())
deduplicated_df = deduplicated_df.loc[eligible_review_mask].reset_index(drop=True)
if VALIDATION_PER_CLASS >= SAMPLES_PER_CLASS:
    raise ValueError('VALIDATION_PER_CLASS must be smaller than SAMPLES_PER_CLASS')

available_per_class = deduplicated_df[TARGET_COLUMN].value_counts().sort_index()
insufficient = available_per_class[available_per_class < SAMPLES_PER_CLASS]
if not insufficient.empty:
    raise ValueError(
        f'Not enough rows for sampling without replacement: {insufficient.to_dict()}'
    )

# Select validation first so it remains identical when SAMPLES_PER_CLASS changes.
validation_raw = deduplicated_df.groupby(
    TARGET_COLUMN, group_keys=False, sort=True
).sample(n=VALIDATION_PER_CLASS, replace=False, random_state=RANDOM_STATE)
training_pool = deduplicated_df.drop(index=validation_raw.index)
train_raw = training_pool.groupby(
    TARGET_COLUMN, group_keys=False, sort=True
).sample(
    n=SAMPLES_PER_CLASS - VALIDATION_PER_CLASS,
    replace=False,
    random_state=RANDOM_STATE + 1,
)
validation_raw = validation_raw.assign(_split='validation')
train_raw = train_raw.assign(_split='train')
balanced_df = pd.concat([train_raw, validation_raw], ignore_index=True)
balanced_df = balanced_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print('Raw rows:', raw_rows)
print('Exact duplicates removed:', exact_duplicates_removed)
print('Missing/blank source reviews removed:', empty_source_reviews_removed)
print('Balanced shape:', balanced_df.shape)
display(balanced_df[TARGET_COLUMN].value_counts().sort_index().to_frame('count'))

## Clean text, merge product metadata, and create `model_input`

In [ ]:
original_review = balanced_df[TEXT_COLUMN].fillna('').astype(str)
original_summary = balanced_df['summary'].fillna('').astype(str)
balanced_df[TEXT_COLUMN] = original_review.map(preprocess_for_transformer)
balanced_df['summary'] = original_summary.map(preprocess_for_transformer)
changed_review_rows = int(original_review.ne(balanced_df[TEXT_COLUMN]).sum())
changed_summary_rows = int(original_summary.ne(balanced_df['summary']).sum())

product_df = pd.read_csv(PRODUCT_PATH, low_memory=False)
product_df['_completeness'] = product_df[['title', 'brand']].notna().sum(axis=1)
product_df = (
    product_df.sort_values('_completeness', ascending=False)
    .drop_duplicates('asin', keep='first')
    [['asin', 'title', 'brand']]
)
balanced_df = balanced_df.merge(
    product_df, on='asin', how='left', validate='many_to_one'
)
balanced_df['title'] = balanced_df['title'].fillna('').astype(str).map(preprocess_for_transformer)
balanced_df['brand'] = balanced_df['brand'].fillna('').astype(str).map(preprocess_for_transformer)

balanced_df['verified_str'] = (
    balanced_df['verified'].map({True: 'yes', False: 'no'}).fillna('unknown')
)
numeric_vote = pd.to_numeric(
    balanced_df['vote'].astype('string').str.replace(',', '', regex=False),
    errors='coerce',
)
missing_vote_rows = int(numeric_vote.isna().sum())
balanced_df['vote_bucket'] = pd.cut(
    numeric_vote.fillna(-1),
    bins=[-2, -0.5, 4.5, 9.5, 49.5, np.inf],
    labels=['missing', '0_to_4', '5_to_9', '10_to_49', '50_plus'],
).astype('string')

def limit_words(value, max_words=SUMMARY_MAX_WORDS):
    return ' '.join(str(value).split()[:max_words])

balanced_df['summary_for_model'] = balanced_df['summary'].map(limit_words)
balanced_df['model_input'] = (
    'Verified: ' + balanced_df['verified_str']
    + ' | Helpful votes: ' + balanced_df['vote_bucket'].fillna('missing')
    + ' | Summary: ' + balanced_df['summary_for_model']
    + ' | Review: ' + balanced_df[TEXT_COLUMN]
)

metadata_coverage = balanced_df['title'].ne('').mean()
print('Changed review rows:', changed_review_rows)
print('Changed summary rows:', changed_summary_rows)
print('Rows retained with missing vote:', missing_vote_rows)
print(f'Product-title coverage: {metadata_coverage:.2%}')
display(balanced_df[[TARGET_COLUMN, 'model_input']].head(3))

## Validate and create the stratified split

In [ ]:
expected_classes = [1, 2, 3, 4, 5]
class_counts = balanced_df[TARGET_COLUMN].value_counts().sort_index()
assert class_counts.index.tolist() == expected_classes
assert class_counts.eq(SAMPLES_PER_CLASS).all()
assert len(balanced_df) == len(expected_classes) * SAMPLES_PER_CLASS
assert not balanced_df.duplicated().any()
assert balanced_df[TEXT_COLUMN].str.strip().ne('').all()
assert balanced_df['model_input'].notna().all()
assert balanced_df['model_input'].str.strip().ne('').all()
assert metadata_coverage > 0.99

train_df = balanced_df.loc[balanced_df['_split'].eq('train')].drop(columns='_split').reset_index(drop=True)
validation_df = balanced_df.loc[balanced_df['_split'].eq('validation')].drop(columns='_split').reset_index(drop=True)
balanced_df = balanced_df.drop(columns='_split')
assert validation_df[TARGET_COLUMN].value_counts().eq(VALIDATION_PER_CLASS).all()
assert train_df[TARGET_COLUMN].value_counts().eq(SAMPLES_PER_CLASS - VALIDATION_PER_CLASS).all()

split_summary = pd.concat({
    'full': balanced_df[TARGET_COLUMN].value_counts().sort_index(),
    'train': train_df[TARGET_COLUMN].value_counts().sort_index(),
    'validation': validation_df[TARGET_COLUMN].value_counts().sort_index(),
}, axis=1)
display(split_summary)
print('Validation: PASSED')

## Save CSV files, metadata, and a publishable ZIP

In [ ]:
# Apply the identical feature pipeline to every test row and preserve its order.
test_df = pd.read_csv(TEST_PATH, low_memory=False)
test_rows = len(test_df)
test_df['_original_order'] = np.arange(test_rows)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].fillna('').astype(str).map(preprocess_for_transformer)
test_df['summary'] = test_df['summary'].fillna('').astype(str).map(preprocess_for_transformer)
test_df = test_df.merge(product_df, on='asin', how='left', validate='many_to_one', sort=False)
test_df = test_df.sort_values('_original_order').reset_index(drop=True)
test_df['title'] = test_df['title'].fillna('').astype(str).map(preprocess_for_transformer)
test_df['brand'] = test_df['brand'].fillna('').astype(str).map(preprocess_for_transformer)
test_df['verified_str'] = test_df['verified'].map({True: 'yes', False: 'no'}).fillna('unknown')
test_numeric_vote = pd.to_numeric(
    test_df['vote'].astype('string').str.replace(',', '', regex=False), errors='coerce'
)
test_df['vote_bucket'] = pd.cut(
    test_numeric_vote.fillna(-1),
    bins=[-2, -0.5, 4.5, 9.5, 49.5, np.inf],
    labels=['missing', '0_to_4', '5_to_9', '10_to_49', '50_plus'],
).astype('string')
test_df['summary_for_model'] = test_df['summary'].map(limit_words)
test_df['model_input'] = (
    'Verified: ' + test_df['verified_str']
    + ' | Helpful votes: ' + test_df['vote_bucket'].fillna('missing')
    + ' | Summary: ' + test_df['summary_for_model']
    + ' | Review: ' + test_df[TEXT_COLUMN]
)
assert len(test_df) == test_rows
assert test_df['_original_order'].eq(np.arange(test_rows)).all()
assert TARGET_COLUMN not in test_df.columns
assert test_df['model_input'].notna().all()
test_df = test_df.drop(columns='_original_order')

balanced_path = OUTPUT_DIR / 'balanced_reviews.csv'
train_path = OUTPUT_DIR / 'train.csv'
validation_path = OUTPUT_DIR / 'validation.csv'
test_path = OUTPUT_DIR / 'test.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'

balanced_df.to_csv(balanced_path, index=False)
train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_train_file': str(TRAIN_PATH),
    'source_product_file': str(PRODUCT_PATH),
    'source_test_file': str(TEST_PATH),
    'samples_per_class': SAMPLES_PER_CLASS,
    'random_state': RANDOM_STATE,
    'validation_per_class': VALIDATION_PER_CLASS,
    'raw_rows': raw_rows,
    'exact_duplicates_removed': exact_duplicates_removed,
    'empty_source_reviews_removed': empty_source_reviews_removed,
    'balanced_rows': len(balanced_df),
    'train_rows': len(train_df),
    'validation_rows': len(validation_df),
    'test_rows': len(test_df),
    'class_distribution': {str(k): int(v) for k, v in class_counts.items()},
    'changed_review_rows': changed_review_rows,
    'changed_summary_rows': changed_summary_rows,
    'product_title_coverage': metadata_coverage,
    'model_input_column': 'model_input',
    'summary_max_words': SUMMARY_MAX_WORDS,
    'missing_vote_policy': 'retain_row_and_encode_as_missing',
    'missing_vote_rows': missing_vote_rows,
    'model_input_fields': ['verified_str', 'vote_bucket', 'summary_for_model', 'reviewText'],
    'columns': balanced_df.columns.tolist(),
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

archive_path = shutil.make_archive(
    str(OUTPUT_DIR),
    'zip',
    root_dir=OUTPUT_DIR,
)
print('Saved files:')
for path in [balanced_path, train_path, validation_path, test_path, metadata_path, Path(archive_path)]:
    print('-', path, f'({path.stat().st_size / 1024**2:.2f} MB)')

## Expected output

With `SAMPLES_PER_CLASS = 50_000` and `VALIDATION_PER_CLASS = 2_000`:

```text
/kaggle/working/bert_balanced_50000_per_class/
├── balanced_reviews.csv   # 250,000 rows
├── train.csv              # 240,000 rows; 48,000 per class
├── validation.csv         # 10,000 rows; 2,000 per class
├── test.csv               # all test rows in original order; no target
└── metadata.json

/kaggle/working/bert_balanced_50000_per_class.zip
```

Upload the folder or ZIP as a Kaggle Dataset, then attach it to `03_roberta_keras.ipynb`.